# Formative 2: Multimodal Data Preprocessing
## User Identity and Product Recommendation System

**Group Members:** Winston (Member 1), Member 2, Member 3  
**Course:** Multimodal Data Preprocessing  

---

### System Overview
This notebook implements a sequential multimodal authentication and product recommendation pipeline:
1. A user's **face** is scanned → verified against known users
2. The user's **voice** is sampled → verified as an approved voiceprint
3. Once authenticated, a **product recommendation** is generated from their social + transaction profile

If any step fails, the system denies access.

In [ ]:
# Install dependencies
# !pip install pandas numpy matplotlib seaborn scikit-learn Pillow librosa soundfile joblib openpyxl

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report, ConfusionMatrixDisplay

plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
print('Environment ready ✓')

---
## Task 1: Data Merge, EDA, and Feature Engineering

### 1.1 Load Datasets

In [ ]:
sp = pd.read_excel('../data/customer_social_profiles.xlsx')
tx = pd.read_excel('../data/customer_transactions.xlsx')

print('Social Profiles shape:', sp.shape)
print('Transactions shape:', tx.shape)

print('\n── Social Profiles Sample ──')
display(sp.head())

print('\n── Transactions Sample ──')
display(tx.head())

### 1.2 Data Cleaning

In [ ]:
print('=== Social Profiles ===')
print('Nulls:', sp.isnull().sum().to_dict())
print('Duplicates:', sp.duplicated().sum())

# Clean social profiles
sp = sp.drop_duplicates()
sp['customer_id'] = sp['customer_id_new'].str.replace('A', '').astype(int)
print(f'After dedup: {sp.shape}')

print('\n=== Transactions ===')
print('Nulls:', tx.isnull().sum().to_dict())
print('Duplicates:', tx.duplicated().sum())

# Fill missing customer ratings with median
median_rating = tx['customer_rating'].median()
tx['customer_rating'] = tx['customer_rating'].fillna(median_rating)
tx['customer_id'] = tx['customer_id_legacy']
tx['purchase_date'] = pd.to_datetime(tx['purchase_date'])
print(f'Null ratings filled with median ({median_rating:.2f})')

### 1.3 Merge Strategy

The `customer_id_new` field uses an 'A' prefix (e.g., `A178`) while `customer_id_legacy` is a plain integer (`178`). After stripping the prefix, we perform an **inner join** to retain only records with a match in both datasets. This ensures our model trains only on customers with full multimodal profiles.

In [ ]:
merged = pd.merge(tx, sp, on='customer_id', how='inner')
print(f'Merged dataset shape: {merged.shape}')
print(f'Columns: {merged.columns.tolist()}')

# Post-merge validation
assert merged['customer_id'].isnull().sum() == 0, 'Null customer IDs!'
assert merged['product_category'].isnull().sum() == 0, 'Null target!'
print('\nPost-merge validation: PASSED ✓')
display(merged.head())

### 1.4 Feature Engineering

In [ ]:
reference_date = merged['purchase_date'].max()
merged['recency_days'] = (reference_date - merged['purchase_date']).dt.days
merged['engagement_x_interest'] = merged['engagement_score'] * merged['purchase_interest_score']
merged['rating_x_amount'] = merged['customer_rating'] * merged['purchase_amount']

le_platform = LabelEncoder()
le_sentiment = LabelEncoder()
merged['platform_encoded'] = le_platform.fit_transform(merged['social_media_platform'])
merged['sentiment_encoded'] = le_sentiment.fit_transform(merged['review_sentiment'])

platform_dummies = pd.get_dummies(merged['social_media_platform'], prefix='platform')
sentiment_dummies = pd.get_dummies(merged['review_sentiment'], prefix='sentiment')
merged = pd.concat([merged, platform_dummies, sentiment_dummies], axis=1)

le_target = LabelEncoder()
merged['product_label'] = le_target.fit_transform(merged['product_category'])

print('Engineered features added:')
print('  recency_days, engagement_x_interest, rating_x_amount')
print('  platform_encoded, sentiment_encoded, one-hot dummies')
print(f'Final shape: {merged.shape}')

merged.to_csv('../features/merged_dataset.csv', index=False)
print('\nSaved → features/merged_dataset.csv')

### 1.5 Exploratory Data Analysis

In [ ]:
print('── Summary Statistics ──')
display(merged[['engagement_score','purchase_interest_score','purchase_amount',
                'customer_rating','recency_days']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA — Merged Customer Dataset', fontsize=14, fontweight='bold')

# Plot 1: Target distribution
ax = axes[0, 0]
merged['product_category'].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Target: Product Category Distribution')
ax.set_xlabel('Category'); ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)

# Plot 2: Engagement score histogram
ax = axes[0, 1]
ax.hist(merged['engagement_score'], bins=20, color='coral', edgecolor='white')
ax.set_title('Engagement Score Distribution')
ax.set_xlabel('Engagement Score'); ax.set_ylabel('Frequency')

# Plot 3: Purchase amount boxplot by category
ax = axes[0, 2]
merged.boxplot(column='purchase_amount', by='product_category', ax=ax)
ax.set_title('Purchase Amount by Category')
ax.set_xlabel('Category'); ax.set_ylabel('Amount ($)')
plt.sca(ax); plt.xticks(rotation=30)

# Plot 4: Correlation heatmap
ax = axes[1, 0]
num_cols = ['engagement_score','purchase_interest_score','purchase_amount',
            'customer_rating','recency_days','engagement_x_interest']
sns.heatmap(merged[num_cols].corr(), ax=ax, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
ax.set_title('Feature Correlation Heatmap')

# Plot 5: Platform pie chart
ax = axes[1, 1]
merged['social_media_platform'].value_counts().plot(
    kind='pie', ax=ax, autopct='%1.1f%%',
    colors=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2'])
ax.set_title('Social Media Platform Mix'); ax.set_ylabel('')

# Plot 6: Sentiment vs avg purchase
ax = axes[1, 2]
merged.groupby('review_sentiment')['purchase_amount'].mean().sort_values().plot(
    kind='barh', ax=ax, color=['#C44E52','#8172B2','#55A868'])
ax.set_title('Avg Purchase Amount by Sentiment')
ax.set_xlabel('Avg Amount ($)')

plt.tight_layout()
plt.savefig('../features/eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()

**EDA Insights:**
- Product categories are roughly balanced (25–35 records each), which means no class weighting is required.
- Engagement score is broadly distributed (60–100), suggesting real variance across users.
- The correlation heatmap shows `engagement_x_interest` has moderate correlation with both parent features as expected — it adds non-linear signal.
- Sentiment does influence average purchase amount, with Positive sentiment linked to slightly higher spend.

---
## Task 2: Image Data Processing

In [ ]:
import sys
sys.path.insert(0, '../scripts')

import importlib.util
spec = importlib.util.spec_from_file_location('task2', '../scripts/task2_image_processing.py')
task2 = importlib.util.load_from_spec(spec)
spec.loader.exec_module(task2)

# Override dirs for notebook context
task2.IMAGES_DIR = '../images'
task2.FEATURES_DIR = '../features'

task2.ensure_images_exist()
images = task2.load_member_images()

In [ ]:
task2.display_sample_images(images)

In [ ]:
task2.display_augmentations(images)

In [ ]:
image_features_df = task2.extract_image_features(images)
print(f'image_features.csv: {image_features_df.shape}')
display(image_features_df[['member_name','expression','augmentation']].head(10))

---
## Task 3: Audio Data Processing

In [ ]:
try:
    import librosa
    spec = importlib.util.spec_from_file_location('task3', '../scripts/task3_audio_processing.py')
    task3 = importlib.util.load_from_spec(spec)
    spec.loader.exec_module(task3)
    task3.AUDIO_DIR = '../audio'
    task3.FEATURES_DIR = '../features'

    task3.ensure_audio_exists()
    audio_data = task3.load_member_audio()
    task3.display_audio_visualizations(audio_data)
    task3.display_augmentation_comparison(audio_data)
    audio_features_df = task3.extract_all_audio_features(audio_data)
    print(f'audio_features.csv: {audio_features_df.shape}')
    display(audio_features_df[['member_name','phrase','augmentation']].head(10))
except ImportError:
    print('[SKIP] Install librosa: pip install librosa soundfile')

---
## Task 4: Model Training and Evaluation

### 4.1 Facial Recognition Model

In [ ]:
import os, joblib
os.makedirs('../models', exist_ok=True)

df_img = pd.read_csv('../features/image_features.csv')
feature_cols_img = [c for c in df_img.columns if c.startswith('hist_') or c.startswith('stat_')]

X_img = df_img[feature_cols_img].values
le_face = LabelEncoder()
y_img = le_face.fit_transform(df_img['member'].values)

scaler_face = StandardScaler()
X_img_s = scaler_face.fit_transform(X_img)

X_tr, X_te, y_tr, y_te = train_test_split(X_img_s, y_img, test_size=0.3, random_state=42, stratify=y_img)

face_model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
face_model.fit(X_tr, y_tr)

y_pred_face = face_model.predict(X_te)
face_acc  = accuracy_score(y_te, y_pred_face)
face_f1   = f1_score(y_te, y_pred_face, average='weighted')
face_loss = log_loss(y_te, face_model.predict_proba(X_te))

print(f'Facial Recognition — Accuracy: {face_acc:.4f}  F1: {face_f1:.4f}  Log-Loss: {face_loss:.4f}')
print(classification_report(y_te, y_pred_face, target_names=le_face.classes_))

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_estimator(face_model, X_te, y_te, display_labels=le_face.classes_, ax=ax, cmap='Blues')
ax.set_title('Facial Recognition — Confusion Matrix')
plt.tight_layout(); plt.savefig('../features/cm_face.png', dpi=120); plt.show()

joblib.dump({'model': face_model, 'scaler': scaler_face, 'le': le_face, 'feature_cols': feature_cols_img},
            '../models/face_model.pkl')
print('Saved → models/face_model.pkl')

### 4.2 Voiceprint Verification Model

In [ ]:
df_aud = pd.read_csv('../features/audio_features.csv')
feature_cols_aud = [c for c in df_aud.columns if c.startswith('mfcc_') or
                    c in ['spectral_rolloff_mean','spectral_rolloff_std',
                           'rms_energy_mean','rms_energy_std','zcr_mean','spectral_centroid_mean']]

X_aud = df_aud[feature_cols_aud].values
le_voice = LabelEncoder()
y_aud = le_voice.fit_transform(df_aud['member'].values)

scaler_voice = StandardScaler()
X_aud_s = scaler_voice.fit_transform(X_aud)

X_tr, X_te, y_tr, y_te = train_test_split(X_aud_s, y_aud, test_size=0.3, random_state=42, stratify=y_aud)

voice_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
voice_model.fit(X_tr, y_tr)

y_pred_voice = voice_model.predict(X_te)
voice_acc  = accuracy_score(y_te, y_pred_voice)
voice_f1   = f1_score(y_te, y_pred_voice, average='weighted')
voice_loss = log_loss(y_te, voice_model.predict_proba(X_te))

print(f'Voiceprint — Accuracy: {voice_acc:.4f}  F1: {voice_f1:.4f}  Log-Loss: {voice_loss:.4f}')
print(classification_report(y_te, y_pred_voice, target_names=le_voice.classes_))

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_estimator(voice_model, X_te, y_te, display_labels=le_voice.classes_, ax=ax, cmap='Blues')
ax.set_title('Voiceprint — Confusion Matrix')
plt.tight_layout(); plt.savefig('../features/cm_voice.png', dpi=120); plt.show()

joblib.dump({'model': voice_model, 'scaler': scaler_voice, 'le': le_voice, 'feature_cols': feature_cols_aud},
            '../models/voice_model.pkl')
print('Saved → models/voice_model.pkl')

### 4.3 Product Recommendation Model

In [ ]:
df_prod = pd.read_csv('../features/merged_dataset.csv')

feature_cols_prod = ['engagement_score','purchase_interest_score','purchase_amount',
                     'customer_rating','recency_days','engagement_x_interest','rating_x_amount',
                     'platform_encoded','sentiment_encoded']
dummy_cols = [c for c in df_prod.columns if c.startswith('platform_') or c.startswith('sentiment_')]
feature_cols_prod += dummy_cols
feature_cols_prod = [c for c in feature_cols_prod if c in df_prod.columns]

X_prod = df_prod[feature_cols_prod].values
le_prod = LabelEncoder()
le_prod.fit(df_prod['product_category'])
y_prod = df_prod['product_label'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_prod, y_prod, test_size=0.25, random_state=42, stratify=y_prod)

candidates = {
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
print('5-Fold Cross Validation:')
for name, clf in candidates.items():
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    scores = cross_val_score(clf, X_tr_s, y_tr, cv=cv, scoring='accuracy')
    results[name] = (scores.mean(), scores.std(), clf, sc)
    print(f'  {name}: {scores.mean():.4f} ± {scores.std():.4f}')

best_name = max(results, key=lambda k: results[k][0])
best_mean, _, best_clf, best_sc = results[best_name]
print(f'\nBest: {best_name} (CV Acc={best_mean:.4f})')

best_sc.fit(X_tr)
best_clf.fit(best_sc.transform(X_tr), y_tr)

X_te_s = best_sc.transform(X_te)
y_pred_prod = best_clf.predict(X_te_s)
prod_acc  = accuracy_score(y_te, y_pred_prod)
prod_f1   = f1_score(y_te, y_pred_prod, average='weighted')
prod_loss = log_loss(y_te, best_clf.predict_proba(X_te_s))

print(f'\nProduct Recommendation — Accuracy: {prod_acc:.4f}  F1: {prod_f1:.4f}  Log-Loss: {prod_loss:.4f}')
print(classification_report(y_te, y_pred_prod, target_names=le_prod.classes_))

fig, ax = plt.subplots(figsize=(7,5))
ConfusionMatrixDisplay.from_estimator(best_clf, X_te_s, y_te, display_labels=le_prod.classes_,
                                       ax=ax, cmap='Blues', xticks_rotation=30)
ax.set_title(f'Product Recommendation ({best_name}) — Confusion Matrix')
plt.tight_layout(); plt.savefig('../features/cm_product.png', dpi=120); plt.show()

joblib.dump({'model': best_clf, 'scaler': best_sc, 'le': le_prod, 'feature_cols': feature_cols_prod},
            '../models/product_model.pkl')
print('Saved → models/product_model.pkl')

### 4.4 Model Metrics Summary

In [ ]:
metrics_df = pd.DataFrame([
    {'Model': 'Facial Recognition', 'Accuracy': face_acc, 'F1 (Weighted)': face_f1, 'Log-Loss': face_loss},
    {'Model': 'Voiceprint Verification', 'Accuracy': voice_acc, 'F1 (Weighted)': voice_f1, 'Log-Loss': voice_loss},
    {'Model': f'Product Recommendation ({best_name})', 'Accuracy': prod_acc, 'F1 (Weighted)': prod_f1, 'Log-Loss': prod_loss},
])

display(metrics_df.set_index('Model').round(4).style.background_gradient(subset=['Accuracy','F1 (Weighted)'], cmap='Greens'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Model Performance Summary', fontsize=13, fontweight='bold')
colors = ['#4C72B0', '#DD8452', '#55A868']
for ax, metric in zip(axes, ['Accuracy', 'F1 (Weighted)']):
    bars = ax.bar(metrics_df['Model'].str.split('(').str[0].str.strip(),
                  metrics_df[metric], color=colors)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, metrics_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../features/model_metrics_summary.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Task 6: System Simulation (Inline Demo)

This simulates the full pipeline inline. For the full CLI experience, run:
```bash
python scripts/task6_system_simulation.py --all
python scripts/task6_system_simulation.py --unauthorized
```

In [ ]:
# Load all models
import joblib
face_bundle  = joblib.load('../models/face_model.pkl')
voice_bundle = joblib.load('../models/voice_model.pkl')
prod_bundle  = joblib.load('../models/product_model.pkl')

MEMBER_NAMES = {'member_1': 'Winston', 'member_2': 'Member 2', 'member_3': 'Member 3'}

def simulate_transaction(member, authorized=True):
    print(f'\n{"="*55}')
    label = f"AUTHORIZED — {MEMBER_NAMES[member]}" if authorized else "UNAUTHORIZED ATTEMPT"
    print(f'  {label}')
    print(f'{"="*55}')

    # Step 1: Face
    from PIL import Image
    img_path = f'../images/{member}/neutral.jpg'
    img = Image.open(img_path).convert('RGB').resize((128, 128))
    if not authorized:
        img_arr = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
        img = Image.fromarray(img_arr)

    def hist_feats(im):
        feats = []
        for ch in im.split():
            h, _ = np.histogram(np.array(ch), bins=16, range=(0,255))
            feats.extend(h / h.sum())
        arr = np.array(im, dtype=np.float32)
        for c in range(3):
            feats.append(arr[:,:,c].mean()); feats.append(arr[:,:,c].std())
        return np.array(feats[:len(face_bundle['feature_cols'])]).reshape(1,-1)

    X_face = face_bundle['scaler'].transform(hist_feats(img))
    pred_face = face_bundle['model'].predict(X_face)[0]
    conf_face = face_bundle['model'].predict_proba(X_face).max()
    pred_member = face_bundle['le'].inverse_transform([pred_face])[0]

    print(f'[STEP 1] Face scan → predicted: {MEMBER_NAMES.get(pred_member, pred_member)} ({conf_face:.1%} confidence)')

    THRESHOLD = 0.50
    if not authorized and conf_face < THRESHOLD:
        print(f'  ✗ Confidence below threshold. ACCESS DENIED.')
        return
    if pred_member != member:
        print(f'  ✗ Identity mismatch. ACCESS DENIED.')
        return
    print(f'  ✓ Face verified')

    # Step 2: Voice (use audio features mean as proxy)
    df_aud = pd.read_csv('../features/audio_features.csv')
    voice_fc = voice_bundle['feature_cols']
    if authorized:
        sample = df_aud[df_aud['member'] == member].head(1)[voice_fc].values
    else:
        sample = np.random.randn(1, len(voice_fc))

    X_voice = voice_bundle['scaler'].transform(sample)
    pred_voice_idx = voice_bundle['model'].predict(X_voice)[0]
    conf_voice = voice_bundle['model'].predict_proba(X_voice).max()
    pred_voice_member = voice_bundle['le'].inverse_transform([pred_voice_idx])[0]
    print(f'[STEP 2] Voice scan → predicted: {MEMBER_NAMES.get(pred_voice_member, pred_voice_member)} ({conf_voice:.1%} confidence)')

    if not authorized and conf_voice < THRESHOLD:
        print(f'  ✗ Voice mismatch. ACCESS DENIED.')
        return
    if pred_voice_member != member:
        print(f'  ✗ Voice mismatch. ACCESS DENIED.')
        return
    print(f'  ✓ Voice verified')

    # Step 3: Product recommendation
    df_prod_data = pd.read_csv('../features/merged_dataset.csv')
    prod_fc = prod_bundle['feature_cols']
    avail_fc = [c for c in prod_fc if c in df_prod_data.columns]
    sample_prod = df_prod_data.sample(1, random_state=42)[avail_fc].values
    if sample_prod.shape[1] < len(prod_fc):
        pad = np.zeros((1, len(prod_fc) - sample_prod.shape[1]))
        sample_prod = np.concatenate([sample_prod, pad], axis=1)
    X_prod_s = prod_bundle['scaler'].transform(sample_prod)
    pred_prod = prod_bundle['model'].predict(X_prod_s)[0]
    conf_prod = prod_bundle['model'].predict_proba(X_prod_s).max()
    product = prod_bundle['le'].inverse_transform([pred_prod])[0]
    print(f'[STEP 3] Product prediction → {product} ({conf_prod:.1%} confidence)')
    print(f'\n   Transaction complete. Welcome, {MEMBER_NAMES[member]}!')
    print(f'  → Recommended product category: {product}')

# Run simulations
simulate_transaction('member_1', authorized=True)
simulate_transaction('member_1', authorized=False)

---
## Summary

| Component | Approach | Output |
|---|---|---|
| Data Merge | Inner join on customer_id (stripped 'A' prefix) | `merged_dataset.csv` |
| Feature Engineering | Recency, cross-product features, one-hot encoding | 20+ features |
| Image Processing | RGB histograms + pixel stats, 3 augmentations | `image_features.csv` |
| Audio Processing | 13 MFCCs + roll-off + energy, 3 augmentations | `audio_features.csv` |
| Face Model | Random Forest (200 trees) | `face_model.pkl` |
| Voice Model | Random Forest (200 trees) | `voice_model.pkl` |
| Product Model | Best of RF / GBM / LR via 5-fold CV | `product_model.pkl` |
| System Simulation | CLI app with auth flow + unauthorized demo | `task6_system_simulation.py` |